In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import skew

# 1. Load the dataset
file_path = 'enriched_collision_data_with_distance.csv'
df = pd.read_csv(file_path)

# 2. Setup Plotting Constants
MAIN_COLOR = "#3498db" # Consistent Blue
FIG_SIZE_UNIT = 5      # Size per subplot for high resolution

def plot_feature_grid_final(features, title, filename, cols=3):
    """
    Final distribution plotting function with intelligent axis scaling.
    """
    if not features: return

    # Filter out columns that don't vary or don't exist
    valid_features = [f for f in features if f in df.columns and df[f].nunique() > 1]
    if not valid_features: return

    rows = int(np.ceil(len(valid_features) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * FIG_SIZE_UNIT, rows * FIG_SIZE_UNIT), squeeze=False)
    fig.suptitle(title, fontsize=26, fontweight='bold', y=1.02)

    axes_flat = axes.flatten()

    for i, col in enumerate(valid_features):
        ax = axes_flat[i]
        data = df[col].dropna()

        # --- LOGIC A: Categorical or Discrete (few unique values) ---
        if data.nunique() < 20 or data.dtype == 'object':
            counts = data.value_counts().sort_index()
            sns.barplot(x=counts.index, y=counts.values, color=MAIN_COLOR, ax=ax)
            if data.nunique() > 5:
                ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
            ax.set_title(f"{col}\n(Categorical)", fontsize=11)

        # --- LOGIC B: Continuous Data ---
        else:
            current_skew = skew(data)
            data_min = data.min()

            # If highly skewed, use Log Scaling
            if abs(current_skew) > 2.0:
                if data_min > 0:
                    # Clean Log Scale
                    sns.histplot(data, kde=True, color=MAIN_COLOR, ax=ax, bins=30, log_scale=True)
                    ax.set_title(f"{col}\n(Log Scale | Skew: {current_skew:.1f})", fontsize=11)
                else:
                    # Log(x + offset) to handle 0s
                    offset = 1 if data_min >= 0 else abs(data_min) + 1
                    transformed_data = np.log10(data + offset)
                    sns.histplot(transformed_data, kde=True, color=MAIN_COLOR, ax=ax, bins=30)
                    ax.set_xlabel(f"Log10(x + {offset})")
                    ax.set_title(f"{col}\n(Log-Transformed | Skew: {current_skew:.1f})", fontsize=11)

            # Normal/Low Skew Data: Use Linear with 1st-99th percentile clipping
            else:
                q_low, q_high = data.quantile(0.01), data.quantile(0.99)
                sns.histplot(data, kde=True, color=MAIN_COLOR, ax=ax, bins=30)
                if q_high > q_low:
                    ax.set_xlim(q_low, q_high)
                ax.set_title(f"{col}\n(Linear | Skew: {current_skew:.1f})", fontsize=11)

        ax.set_ylabel("Frequency")
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # Clean up empty subplot slots
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis('off')

    plt.tight_layout(pad=3.0)
    plt.savefig(filename, bbox_inches='tight', dpi=1200)
    plt.close()
    print(f"Generated: {filename}")

# 3. Categorization & Execution
all_cols = df.columns.tolist()

# Group 1: General Accident Info & Nearest Distances
dist_basic = all_cols[:32] + ['Severity']
plot_feature_grid_final(dist_basic, "Accident & Distance Distributions", "dist_log_aware_basic.png")

# Group 2: Semantic Segmentation (Percentages)
seg_features = [c for c in all_cols if c.startswith('seg_')]
plot_feature_grid_final(seg_features, "Semantic Segmentation Distributions", "dist_log_aware_segmentation.png")

# Group 3: Image Quality Analysis
img_features = [c for c in all_cols if c.startswith(('luminance_', 'sat_', 'val_', 'color', 'warmth', 'laplacian', 'tenengrad', 'edge_', 'local_', 'dark_', 'white_', 'specular_', 'vertical_'))]
plot_feature_grid_final(img_features, "Image Feature Distributions", "dist_log_aware_image.png")

# Group 4: Object Detection (Batched in 30s)
obj_features = [c for c in all_cols if c.startswith('obj_')]
obj_batches = [obj_features[i:i + 30] for i in range(0, len(obj_features), 30)]
for idx, batch in enumerate(obj_batches):
    plot_feature_grid_final(batch, f"Object Detection Distributions (Part {idx+1})", f"dist_log_aware_objects_{idx+1}.png")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')